## Topic: LangGraph Core Components

### Agenda:

- 1. Introduction to LangGraph Core Components

- 2. LLM Workflow

- 3. Why Does LangGraph Use Graphs?


- 4.  State — The Shared Memory of a Graph

- 5.  StateGraph — The Graph Builder

- 6.  Nodes — Units of Work

- 7.  Edges — Connections Between Nodes

- 8.  START and END Nodes

- 9.  Reducers — How State Updates Are Merged

- 10. LangGraph Execution Model

- 11. Complete LangGraph Architecture

- 12. Key Takeaways


### 1. Introduction to LangGraph Core Components

- Definition of LangGraph:
    - LangGraph is an orchestration framework for building intelligent, stateful, multi-step and controllable applications of LLM workflow.


    - It enables advanced features like parallelism, loops, branching, memory and resumability __ making it ideal for agentic and production-grade AI applications.

    - It models our logic as a graph of nodes(tasks) and edges(routing) instead of a linear chian.



- Key Rule of Thump:
    - LangGraph Core Components are the building blocks you use to construct graph-based agent workflows.

    - LangChain Components (Bricks):
        - ChatOpenAI, PromptTemplate, Retriever, Tools, Parsers

    - LangGraph Components (Blueprint):
    - State, StateGraph, Nodes, Edges, Conditional Edges, Reducers, Checkpointers, Compile


    - Together:
        - LangGraph blueprint orchestrates LangChain bricks.


### 2. LLM Workflow

- 1. Workflow:
=================
- 
    - Series of tasks for execute of right order to achieve goal.

- 2. LLM Workflow:
=================
- 
    - LLM workflow are a step by step process using which we can build complex LLM application.

    - Each step in a workflow performs a distinct task __ such as prompting, reasoning, tool calling, memory access, or decision-making.

    - Workflows can be linear, parallel, branched, or looped, allowing for complex behaviours like retries, multi-agent communication, or tool-augmented reasoning.

    - Each application has a unique workflow.
    
    - Key Rule of Thump:    
        - If a workflow for execute of tasks use LLM that is called LLM Workflow.


- Common Workflows
=================
- 
     - 1. Prompt Chaining:
        - We use(call) multiple time LLM in series format.
        - eg: topic --> LLM --> outline --> LLM --> details report
        - use when we have complex task that divide into sub task.

    - 2. Routing:
        - eg:
        -  Input(query) --> LLM call(router)
        -                            ---> LLM call 1
        -                            ---> LLM call 2           ----> output
        -                            ---> LLM call 3

        - router call decide, which LLM is execute.
    
    - 3. Parallelization:
        - eg: Youtube content Checker
        -  Input(query) ---> LLM call 1 (check the youtube community guideline) 
        -               ---> LLM call 2 (check the misinformation)                    ---> Aggregator ----> output
        -               ---> LLM call 3 (check the sexual content)          
       
        - Based on the query at a time execute all  LLM.
        - here, the sub task are predefine that each LLM are execute.

    - 4. orchestrator Workers
        - eg: Research Report Generator

        -  Input(query) --> LLM call(orchestrator)
        -                    ---> LLM call 1             ---> Synthesizer ----> output
        -                    ---> LLM call 2           
        -                    ---> LLM call 2      


        - orchestrator LLM automatically decide which LLM is execute at run time.
        - here, we don't know the nature of sub task that are LLM execute.
        - Depending on the query, the orchestrator LLM decides which LLM should be called.


    - 5. Evaluator Optimizer:
        - There are two LLM. first LLM (Generator LLM) are use to Create Generator content and the second LLM (Evaluator LLM) are use for Evaluate of the first LLM response and also provide the Feedback.

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│        WHY LEARN LANGGRAPH COMPONENTS FIRST?                │
│                                                             │
│  1. EVERY WORKFLOW USES THE SAME PARTS                      │
│     Sequential, Parallel, Conditional, Iterative: all are   │
│     just different arrangements of State, Nodes, and Edges. │
│                                                             │
│  2. MOST BUGS COME FROM STATE MISUNDERSTANDING              │
│     "Why was my list overwritten?" → You needed a reducer.  │
│                                                             │
│  3. ROUTING LOGIC IS THE HEART OF AGENTS                    │
│     Conditional edges are how agents "decide."              │
│                                                             │
│  4. PERSISTENCE, MEMORY, AND HITL ALL DEPEND ON CHECKPOINTS │
│     Module 3 and 4 topics build directly on these.          │
│                                                             │
│  5. YOU CAN READ ANY LANGGRAPH CODE                         │
│     Once you know the 10 components, every example in the   │
│     docs becomes readable.                                  │
└─────────────────────────────────────────────────────────────┘


""" 

### 3. Why Does LangGraph Use Graphs?

In [ ]:
"""
- 1. The Problem with Linear Chains:
=====================================

LangChain Chain (Linear):

  Step A → Step B → Step C → Done

    - Cannot go back to Step A
    - Cannot skip Step B based on a condition
    - Cannot run Step A and Step B in parallel
    - Cannot loop until a condition is met

    
- 2. The Solution: Graphs:
============================
LangGraph (Graph):

         START
           │
           ▼
      ┌─────────┐
      │ Node A  │
      └────┬────┘
           │
      ┌────▼────┐
      │ Router  │ ← Conditional: which path?
      └──┬───┬──┘
     Yes │   │ No
         ▼   ▼
    ┌──────┐ ┌──────┐
    │Node B│ │Node C│
    └──┬───┘ └──┬───┘
       │        │
       └───┬────┘
           ▼
      ┌─────────┐
      │ Node D  │
      └────┬────┘
           │
           ▼
          END

- Key Note: 
    - Can branch (conditional edges)
    - Can loop (edges back to earlier nodes)
    - Can run in parallel (multiple edges from one node)
    - Can merge (multiple edges into one node)


"""

In [ ]:
""" 
Visual Graph Representation: 
============================

                 ┌───────────┐
                 │   START   │
                 └─────┬─────┘
                       │
                       ▼
          ┌────────────────────────┐
          │    create_greeting     │
          │         NODE           │
          └───────────┬────────────┘
                      │
                      ▼
                 ┌───────────┐
                 │    END    │
                 └───────────┘

=======================================================================
                 
                 
                  ┌─────────────┐
                  │    START    │
                  └──────┬──────┘
                         ↓
                  ┌─────────────┐
                  │    NODE     │
                  │  Process    │
                  └──────┬──────┘
                         ↓
                  ┌─────────────┐
                  │    STATE    │
                  │   Update    │
                  └──────┬──────┘
                         ↓
                  ┌─────────────┐
                  │    EDGE     │
                  │   Routing   │
                  └──────┬──────┘
                         ↓
                    ┌───────┐
                    │  END  │
                    └───────┘

                    
- Key Rule of Thump:
    - Graph = State + Nodes + Edges


"""

### 4.  State — The Shared Memory of a Graph

### 5.  StateGraph — The Graph Builder

### 6.  Nodes — Units of Work

### 7.  Edges — Connections Between Nodes

In [ ]:
""" 
    - STATE, NODES, EDGES and GRAPH: 
    ==================================
    
            ┌──────────────────────────────────────────────────────────────┐
            │                                                              │
            │  STATE                                                       │
            │  Shared data that moves through the workflow                 │
            │  Example: messages, user query, retrieved documents          │
            │                                                              │
            │       │                                                      │
            │       ▼                                                      │
            │  NODES                                                       │
            │  Python functions that perform work                          │
            │  Example: call_llm(), search_documents(), grade_answer()     │
            │                                                              │
            │       │                                                      │
            │       ▼                                                      │
            │  EDGES                                                       │
            │  Define where the workflow goes next                         │
            │  Example: agent → tools → agent                              │
            │                                                              │
            │       │                                                      │
            │       ▼                                                      │
            │  GRAPH                                                       │
            │  The complete workflow created with StateGraph               │
            │                                                              │
            └──────────────────────────────────────────────────────────────┘

"""

### 8.  Reducers — How State Updates Are Merged

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│               THE 10 LANGGRAPH CORE COMPONENTS                   │
│                                                                  │
│  DATA                                                            │
│    1. State            → Shared data schema (TypedDict/Pydantic) │
│    2. Reducers         → Rules for merging state updates         │
│                                                                  │
│  WORK                                                            │
│    3. Nodes            → Python functions that do the work       │
│                                                                  │
│  FLOW                                                            │
│    4. Edges            → Fixed connections (A → B)               │
│    5. Conditional Edges→ Dynamic routing (A → B or C)            │
│    6. START / END      → Entry and exit points                   │
│                                                                  │
│  BUILD & RUN                                                     │
│    7. StateGraph       → The builder; compile() makes it runnable│
│    8. Execution        → invoke(), stream(), batch()             │
│                                                                  │
│  PRODUCTION                                                      │
│    9. Checkpointer     → Persistence, memory, resume, rewind     │
│   10. Send / Command / interrupt → Map-reduce, routing, HITL     │
└──────────────────────────────────────────────────────────────────┘


"""

### 9.  START and END Nodes

### 10. LangGraph Execution Model

### 11. Complete LangGraph Architecture

### 12. Key Takeaways